<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Modern AI: LLMs & Generative AI** · Week 4 · Agents, Generative AI and Responsible Deployment · Lesson 21 of 24

# Generative AI Beyond Text: Images, Audio and Diffusion

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

**API key:** this lesson calls an AI model. Add your key with the **key icon (Secrets)** in the left bar, switch on **Notebook access**, and never paste a key into the code.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative).

In [ ]:
# Install the libraries this lesson needs (takes a few seconds).
%pip install -q anthropic

**Learning objective:** Explain how diffusion models generate images, train a tiny diffusion model yourself, send an image to a multimodal LLM, and recognise the main tools and risks of generated media.

## The generative AI family

| Task | Main technique | Examples of systems |
|---|---|---|
| Text → text | Transformer language models | Claude, Gemini, GPT, Llama |
| Text → image | **Diffusion** models guided by text | Stable Diffusion, FLUX, Imagen |
| Image → text | Multimodal LLMs ("vision") | Claude, Gemini, GPT can read images |
| Speech → text | Transformer speech models | Whisper (open, many languages) |
| Text → speech, music, video | Transformers and diffusion | Many commercial and open systems |

Text and images are linked by models such as **CLIP**, trained so that an image and its caption get **similar embeddings** (Lesson 13). That shared embedding space is how a text prompt can guide image generation.

## How diffusion works

1. **Forward process (easy):** take a real image and add a little random noise, many times, until it is pure noise.
2. **Train a network** to look at a noisy image and the noise level, and predict the noise that was added.
3. **Generate (reverse process):** start from pure noise and repeatedly remove the predicted noise. A clean, new image appears.

Image models do this with millions of pixels and a text prompt as extra input. We will do it with 2-D points so it trains in about a minute on a CPU — the maths is the same.

## The data: points shaped like two moons

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
data = torch.tensor(make_moons(4000, noise=0.05, random_state=0)[0], dtype=torch.float32)
data = (data - data.mean(0)) / data.std(0)

T = 100
betas = torch.linspace(1e-4, 0.1, T)                 # how much noise is added at each step
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)             # how much of the original survives after t steps

def add_noise(x0, t):
    noise = torch.randn_like(x0)
    return alpha_bar[t].sqrt()[:, None] * x0 + (1 - alpha_bar[t]).sqrt()[:, None] * noise, noise

print(f"signal left after 10 steps: {alpha_bar[9].sqrt():.2f}, after 100 steps: {alpha_bar[-1].sqrt():.3f}")

After 100 steps almost nothing of the original is left — just noise.

## Train the denoiser

In [ ]:
denoiser = nn.Sequential(nn.Linear(3, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(),
                         nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(denoiser.parameters(), lr=2e-3)

for step in range(3000):
    x0 = data[torch.randint(0, len(data), (512,))]
    t = torch.randint(0, T, (512,))
    xt, noise = add_noise(x0, t)
    pred = denoiser(torch.cat([xt, (t / T)[:, None]], dim=1))   # input: noisy point + noise level
    loss = ((pred - noise) ** 2).mean()                          # learn to predict the noise
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0 or step == 2999:
        print(f"step {step:4d}  loss {loss.item():.3f}")

## Generate new points from pure noise

In [ ]:
@torch.no_grad()
def sample(n=2000):
    x = torch.randn(n, 2)                                         # start: pure noise
    for t in reversed(range(T)):
        eps = denoiser(torch.cat([x, torch.full((n, 1), t / T)], dim=1))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)         # a little fresh noise
    return x

generated = sample()
noise_only = torch.randn(2000, 2)
dist = lambda pts: torch.cdist(pts, data).min(dim=1).values.median().item()
print(f"median distance to the real data — generated: {dist(generated):.3f}, pure noise: {dist(noise_only):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, pts, title in zip(axes, [data[:2000], noise_only, generated], ["real data", "start: noise", "generated"]):
    ax.scatter(pts[:, 0], pts[:, 1], s=2); ax.set_title(title); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show()

The generated points form the two moons again, far closer to the real data than random noise — **new** points, not copies. An image diffusion model does exactly this, with a large network (often a transformer) predicting the noise in every pixel, guided by the text prompt's embedding.

## Images as input: multimodal LLMs

Modern LLMs can also **read** images — receipts, charts, screenshots, handwritten forms. You send the image as base64 data alongside your question:

In [ ]:
import importlib.util, os, subprocess, sys

if importlib.util.find_spec("anthropic") is None:          # install once per Colab session
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)

try:
    from google.colab import userdata                     # Colab: key saved in the Secrets panel (key icon)
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass                                                    # not in Colab: set ANTHROPIC_API_KEY in your environment
except Exception:
    print("Add a secret called ANTHROPIC_API_KEY in Colab's Secrets panel, then run this cell again.")

import anthropic

client = anthropic.Anthropic()        # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-5-5"            # fast and low-cost; check docs.claude.com for current model names

def ask(prompt, system=None, max_tokens=500):
    extra = {"system": system} if system else {}
    response = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                      messages=[{"role": "user", "content": prompt}], **extra)
    return response.content[0].text

In [ ]:
import base64
import io

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["Mon", "Tue", "Wed", "Thu", "Fri"], [12, 18, 9, 22, 30], color="teal")
ax.set_title("Mobile money payments per day")
buf = io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
image_b64 = base64.standard_b64encode(buf.getvalue()).decode()

response = client.messages.create(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user", "content": [
        {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_b64}},
        {"type": "text", "text": "Which day had the most payments? Answer in one sentence."},
    ]}],
)
print(response.content[0].text)

Images cost tokens too (roughly in proportion to their size), so resize big photos before sending.

## Speech

Open speech-recognition models such as **Whisper** transcribe dozens of languages; quality differs a lot between languages, so test with real recordings of your users — including Kinyarwanda and code-switching between Kinyarwanda, English and French.

## Risks of generated media

- **Deepfakes:** fake images, voices or videos of real people used for fraud, harassment or political lies. Never generate media of a real person without their consent.
- **Copyright and style:** models are trained on others' work; follow the licence and do not imitate living artists' work or brands.
- **Labelling:** say clearly when an image, voice or text is AI-generated. Some systems add invisible watermarks.
- **Bias:** generated people often follow stereotypes from the training data — check outputs for who is shown and how.

## Key terms

- **Diffusion model** — learns to remove noise step by step, generating data from pure noise.
- **Forward / reverse process** — adding noise / removing noise.
- **Noise schedule** — how much noise is added at each step (betas).
- **CLIP** — a model that puts images and text in the same embedding space.
- **Multimodal model** — a model that accepts several kinds of input, such as text and images.
- **Deepfake** — realistic fake media of a real person.

**Takeaway:** Diffusion models generate by learning to remove noise; multimodal LLMs read images and other media; generated media must be used with consent, labels and care.

## Try it yourself

1. Train the denoiser for only 300 steps. What do the generated points look like?
2. Change `T` to 20. Is generation faster? Is it worse?
3. Replace the moons with `make_circles` from scikit-learn and train again.
4. Send a photo of a printed receipt (no personal data) to the model and ask it to extract the total as JSON.

---
Finished? Go back to [Modern AI: LLMs & Generative AI on WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*